## Mean-Variance Optimization with risky asset and cash

## Purpose:

* Reinforcement learning is applied to the problem of optimal allocation
* The least squares policy iteration (LSPI) algorithm is applied to a Monte Carlo simulation of a stock's price movements, constructing a basis over the state-action space using B-spline basis functions at each time period.
* The optimal Q-function is approximated with a dynamic programming approach, and this is shown to approach the exact solution

In [ ]:
import numpy as np
import pandas as pd

from scipy.interpolate import interp1d
from scipy.stats import norm

from bspline import Bspline, splinelab

import matplotlib.pyplot as plt
%matplotlib inline

## Parameters for MC simulation of stock prices

In [ ]:
S0 = 100 # Initial stock price
mu = 0.03 # Drift
sigma = 0.20 # Volatility
r = 0.01 # Risk-free rate
M = 1 # Maturity

T = 30 # Number of time steps
delta_t = M / T # Time interval

N_MC = 2000 # Number of paths

gamma = np.exp(- r * delta_t) # Discount factor
lmbda = 10.0 # Risk aversion
alpha = 1.0 # Learning rate

## Create dataset

In [ ]:
# Stock price. Give the frame a float dtype up front: an empty frame is dtype
# object, and the object column then travels all the way into scipy's
# interpolator further down, which refuses it.
S = pd.DataFrame(np.zeros((N_MC, T+1)), index=range(1, N_MC+1), columns=range(T+1))
S.loc[:, 0] = S0

# Standard normal random numbers
RN = pd.DataFrame(np.random.randn(N_MC, T), index=range(1, N_MC+1), columns=range(1, T+1))

for t in range(1, T+1):
    S.loc[:, t] = S.loc[:, t-1] * np.exp((mu - 1/2 * sigma**2) * delta_t + sigma * np.sqrt(delta_t) * RN.loc[:, t])
    
    # Or, using Euler for alpha-stable distributions:
    #S.loc[:, t] = S.loc[:, t-1]*(1 + mu*delta_t + sigma *S.loc[:,t-1]**(alpha-1)*np.sqrt(delta_t) * RN.loc[:,t])

# Plot 10 paths
step_size = N_MC // 10
idx_plot = np.arange(step_size, N_MC, step_size)
plt.plot(S.T.iloc[:, idx_plot])
plt.xlabel('Time Steps')
plt.title('Stock Price Sample Paths')
plt.show()


$$C_t = -R_t= -r_t  + \lambda (r_t-\mu)^2  =  -(1-u_t)r_f - u_t  \phi_t  +  \lambda u_t^2 Var(\phi_t | S_t)$$ 

In [ ]:
def negative_reward(mu, var, a, rf, lmbda):
    C = -(1-a)*rf - a*mu + lmbda*(a**2)*var
    return C

##  Define spline basis functions  

In [ ]:
X = S # set the wealth (but not the total wealth) of the portfolio to be the stock price
p = 4 # order of spline (as-is; 3 = cubic, 4: B-spline)
ncolloc = 16
a_min = -1
a_max = 1

In [ ]:
def get_basis_functions(X_min, X_max, a_min, a_max, ncolloc, p=3):
    tau_x = np.linspace(X_min, X_max, ncolloc)  # These are the sites to which we
    tau_a = np.linspace(a_min, a_max, ncolloc)  # would like to interpolate

    # k is a knot vector that adds endpoints repeats as appropriate for a spline of order p
    # To get meaninful results, one should have ncolloc >= p+1
    k_x = splinelab.aptknt(tau_x, p)
    k_a = splinelab.aptknt(tau_a, p)
                             
    # Spline basis of order p on knots k
    basis_x = Bspline(k_x, p)
    basis_a = Bspline(k_a, p)
    return basis_x, basis_a

### Make data matrices with feature values

"Features" here are the values of basis functions at data points
The outputs are 3D arrays of dimensions num_tSteps x num_MC x num_basis

In [ ]:
num_t_steps = T + 1
num_basis =  ncolloc**2 

data_mat_t = np.zeros((num_t_steps, N_MC, num_basis ))

In [ ]:
x = X.values[:, 0]
np.shape(x)

## Compute the optimal Q-function with the DP approach 

Coefficients for expansions of the optimal Q-function $Q_t^\star\left(X_t,a_t^\star\right)$ are solved by

$$W_t=\mathbf S_t^{-1}\mathbf M_t$$

where $\mathbf S_t$ and $\mathbf M_t$ are matrix and vector respectively with elements given by

$$S_{nm}^{\left(t\right)}=\sum_{k=1}^{N_{MC}}{\Phi_n\left(X_t^k,a_t^k\right)\Phi_m\left(X_t^k,a_t^k\right)}\quad\quad M_n^{\left(t\right)}=\sum_{k=1}^{N_{MC}}{\Phi_n\left(X_t^k,a_t^k\right)\left(C\left(X_t^k,a_t^k,X_{t+1}^k\right)+\gamma\min_{a_{t+1}\in\mathcal{A}}Q_{t+1}^\star\left(X_{t+1}^k,a_{t+1}^k\right)\right)}$$

Define function *function_S* and *function_M* to compute the value of matrix $\mathbf S_t$ and vector $\mathbf M_t$.

In [ ]:
def function_S_vec(t, data_mat_t):
    # Compute the matrix S_{nm} 
    X_mat = data_mat_t[t, :, :]
    num_basis_funcs = X_mat.shape[1]    
    S_mat = np.dot(X_mat.T, X_mat)
    return S_mat

def function_M_vec(t, Q, R, data_mat_t, gamma=0.1):
    X_mat = data_mat_t[t,:,:]
    tmp = R + gamma * np.min(Q[:, t+1])  # note that the second argument in Q is t+1
    M = np.dot(X_mat.T, tmp)
    return M

### Least Squares Policy Iteration
Call *function_S* and *function_M* for $t=T-1,...,0$ together with basis function $\Phi_n\left(X_t,a_t\right)$ to compute optimal action Q-function $Q_t^\star\left(X_t,a_t^\star\right)=\sum_n^N{\omega_{nt}\Phi_n\left(X_t,a_t^\star\right)}$ backward recursively with terminal condition $Q_T^\star\left(X_T,a_T=0\right)=0$.



#### Initialize data structures

In [ ]:
mu = np.zeros(T)
var = np.zeros(T)

# optimal action
a_opt = np.zeros((N_MC, T))
a_star = pd.DataFrame(np.zeros((N_MC, T+1)), index=range(1, N_MC+1), columns=range(T+1))
a_star.iloc[:, -1] = 0

# optimal Q-function with optimal action
max_Q_star = np.zeros((N_MC, T))

a_mean = []

#### initialize actions in feasible region [0, a_max]

In [ ]:
grid_size_x = 20
grid_size_a = 500 

# set up a small grid, sufficiently nested inside the support of the basis functions
a_min_prime = a_min + 0.1
a_max_prime = a_max - 0.1

a_grid = np.linspace(a_min_prime, a_max_prime, grid_size_a)

Q_star = np.zeros((N_MC, T))
tau = 1e-6
a = a_min_prime + (a_max_prime - a_min_prime) * np.random.rand(N_MC)

There are *grid_size_x* $\times$ *grid_size_a* inner grid points and 256 basis functions. Let $(k,l)$ denote the indices of the smaller grid $\Omega^h$. Let $(i,j)$ denote the indices of the knot points of the basis functions. If you sum over the last index, then you can check the interpolation of ones
$$f(x_k,a_l)=\sum_{ij} \Phi(x_k,a_l)f_{ij}$$
check if $f_{ij}=1$ everywhere
$$f(x,a)=\sum_{ij} \Phi(x,a)1 =1, \forall x,a ?$$

#### The backward loop

*Make sure to rerun cells under the heading* **'Initialize data structures'** *to reset the initial conditions before running this cell*

In [ ]:
for t in np.arange(T - 2, 0, -1):
    error = tau
    q_prev = np.zeros(N_MC)  
    ret = (S.loc[:, t+1] - S.loc[:, t]) / S.loc[:, t]
    mu[t] = np.mean(ret)
    var[t] = np.var(ret)
    count = 0
    max_iter = 100
    x = X.values[:, t]
    basis_x, basis_a = get_basis_functions(np.min(x)-10, np.max(x)+10, a_min, a_max, ncolloc, p)
    x_grid = np.linspace(np.min(x), np.max(x), grid_size_x)
    Phi_mat = np.array([[np.kron(basis_x(x_grid[i]), basis_a(a_grid[j])).reshape(num_basis, 1) 
                         for i in range(grid_size_x)] for j in range(grid_size_a)])[:, :, :, 0]   
        
    while (np.abs(error)>=tau) and (count<max_iter):
       
        R = negative_reward(mu[t], var[t], a, r, lmbda)
        data_mat_t[t, :, :] = np.array([np.kron(basis_x(x[i]), basis_a(a[i])).reshape(num_basis, 1) for i in range(N_MC)])[:, :, 0]
        
        # Check partition of unity
        h = np.dot(data_mat_t[t],np.ones(np.shape(data_mat_t[t])[1]))
        if np.sum(h) != N_MC:
            print("error: loss of partition of unity")
        S_t = function_S_vec(t, data_mat_t) 
        M_t = function_M_vec(t, Q_star, R, data_mat_t, gamma)
        W_t = np.dot(np.linalg.pinv(S_t), M_t)
       
        # Compute Q_t matrix over small grid
        # Phi_mat is the matrix for interpolating over the smaller x * a grid
        Q_t = np.dot(Phi_mat, W_t) # gridded Q_t
        
        print("residual error: ||r||=||Sw-M||")
        print(np.linalg.norm(np.dot(S_t, W_t) - M_t))
        
        # Find the optimal action on the small grid
        a_idx =  np.argmin(Q_t, axis=0)
        # Need to interpolate over X 
        a_star_ = np.zeros(grid_size_x, dtype='float64')
        
        for j in range(grid_size_x):
            a_star_[j] = a_grid[a_idx[j]]
    
        f = interp1d(x_grid, a_star_, kind='cubic')
        a_prime = f(x)
        
        # Only update the actions along the paths where the neg. reward is lowered
        R_prime = negative_reward(mu[t], var[t], a_prime, r, lmbda)
        idx = np.where(R_prime>R)
        a_prime[idx] = a[idx]
        a = a_prime
        a_mean.append(np.mean(a))
        Q_star[:,t] = np.dot(data_mat_t[t, :, :], W_t)
        error = np.linalg.norm(Q_star[:, t] - q_prev)
        
        q_prev = np.copy(Q_star[:, t])
        
        print('count, Q error, E[R], E[a]')
        print(count, error, np.mean(R), np.mean(a))
        count += 1 
       
    a_opt[:, t] = a  

In [ ]:
a_star_exact = []
a_star_approx = []
for t in range(1, T-1):
    a_star_exact.append((mu[t] - r) / (2 * lmbda * var[t]))
    a_star_approx.append(np.mean(a_opt[:, t]))

In [ ]:
plt.plot(a_star_approx)
plt.plot(a_star_exact, color='red')
plt.xlabel('time')
plt.ylabel('action');

In [ ]:
a_star_exact = (mu[t] - r) / (2 * lmbda * var[t])

In [ ]:
a_star_exact

In [ ]:
negative_reward(mu[t], var[t], a_star_exact, r, lmbda)

In [ ]:
a_ = np.arange(-1, 1, 0.01)
plt.plot(a_, negative_reward(mu[t], var[t], a_, r, lmbda))
plt.xlabel('S')
plt.ylabel('R');

In [ ]:
plt.plot(a_mean)
plt.plot(x_grid, a_star_)
plt.plot(x_grid, np.ones(len(x_grid)) * a_star_exact, color='black')
plt.ylim([-1, 1])

optimal action depends on lambda